# 线性模型学以致用

本次将重新定义一个类似模型，并参考书中做法重新跑一便实验，以达到加深对模型理解，回顾所学知识及寻找和发现自己的不足之处
按老师要求假设公式为
$y= w_1x_1+w_2x_2+w_3x_3+...+w_{99}x_{99}+w_{100}x_{100}+b_1+b_2+...+b_{100}+\epsilon$
## 按此公式生成数据集

In [1]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

#第一版:这里生成100个随机w和b，样本生成一万个
生成十万个样本

In [2]:
true_w = torch.randn(100)
true_b = torch.randn(100)
print(true_w)
print(true_b)


tensor([-0.0138, -1.1542,  0.4269, -0.7163,  0.7944, -0.3133, -0.8049,  0.0815,
        -1.0893, -0.9322, -0.3525, -0.3845, -0.1417, -0.0485, -0.6260, -0.5856,
         1.0104, -1.2012, -0.4837,  0.4860,  0.1515,  0.7371, -1.0569,  1.4312,
        -1.1051,  0.2920, -1.2776,  0.8077,  0.5414,  0.2747,  0.5204,  1.1319,
        -0.1913,  1.3294,  0.9934, -0.1366, -2.0902,  0.9641,  0.4653, -1.7711,
        -0.6393, -0.0835, -1.1170,  1.7375,  1.7405,  0.6244, -0.3248,  1.0670,
        -0.2164, -2.0466, -0.2075, -1.2917,  0.9548,  1.1475, -0.2182, -0.4196,
        -0.3583,  0.7271,  0.2292, -0.3609,  0.4784, -1.2855, -1.7409,  0.9422,
         0.8867,  0.3583, -1.5552, -0.4309, -0.1943, -1.1912,  0.6957, -1.9194,
        -1.3376, -1.8517,  0.0139,  0.2225,  0.3967, -0.7787,  0.9242, -0.2608,
        -0.2607, -2.0059,  0.6630, -0.2437,  0.4085, -0.2379, -0.3385, -1.2286,
         0.4751,  0.6625,  0.2592, -0.1481, -0.7761, -0.5209,  0.6523, -1.6344,
         0.0823,  0.2883,  1.2835, -0.71

## 读取数据集
由于原函数不适配，这里参考3.2重写一个
遇到如下报错
```
Cell In[37], line 5
      3 true_w
      4 true_b
----> 5 features, labels = d2l.synthetic_data(true_w, true_b, 1000)

File ~/miniconda3/envs/d2l/lib/python3.9/site-packages/d2l/torch.py:142, in synthetic_data(w, b, num_examples)
    138 """Generate y = Xw + b + noise.
    139 
    140 Defined in :numref:`sec_linear_scratch`"""
    141 X = d2l.normal(0, 1, (num_examples, len(w)))
--> 142 y = d2l.matmul(X, w) + b
    143 y += d2l.normal(0, 0.01, y.shape)
    144 return X, d2l.reshape(y, (-1, 1))

RuntimeError: The size of tensor a (1000) must match the size of tensor b (10) at non-singleton dimension 0
```
原因：
`true_b` 被错误定义为**多维度 / 多元素的张量**（比如长度为 10 的向量），但 `synthetic_data` 函数中 `b` 应该是**单个标量值**（比如 `4.2`），而非向量 / 矩阵。

解决方法:
这里X*w生成的是一个行为num_examples即10000，列为w长度即100，的张量
需要将b扩展为相应形式，将生成num_examples行，每行为b的b_expanded

In [3]:
def synthetic_data1(w, b, num_examples):  #@save
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    #这里遇到xw与b无法广播，将生成num_examples行，每行为b的b_expanded
    b_expanded = b.repeat(num_examples // len(b) + 1)[:num_examples]
    Xw = torch.matmul(X, w) 
    y = Xw + b_expanded
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))
features, labels = synthetic_data1(true_w, true_b, 100000)

这里按照书读取数据集,将样本打包为十个一组

In [4]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

#由于特征与样本都增加，这里考虑适度提高batch_size的大小，即将更多样本打包在一起如32个样本

加到64样本打包

In [5]:
batch_size = 64
data_iter = load_array((features, labels), batch_size)

In [6]:
next(iter(data_iter))

[tensor([[ 1.0850, -0.0103,  0.9672,  ..., -0.2445,  0.2231, -0.6719],
         [-1.0353,  1.2118,  1.0357,  ...,  0.6868,  0.9176,  0.7831],
         [ 0.1264, -0.2621, -0.8784,  ..., -1.8812,  0.5364,  0.6571],
         ...,
         [-1.3266, -0.3841, -0.6936,  ...,  0.0631, -2.2987,  0.1094],
         [ 1.9774,  2.3681,  0.5284,  ..., -0.6068, -0.2185,  0.9039],
         [-1.3039,  1.4942, -0.4454,  ..., -1.0932, -1.2101,  1.7916]]),
 tensor([[ 18.8250],
         [  1.9338],
         [ -5.6760],
         [  2.0886],
         [-10.7396],
         [-11.7603],
         [ -0.9357],
         [ -0.0867],
         [ -5.8259],
         [ -5.2175],
         [ 15.5967],
         [  2.2102],
         [  7.4359],
         [ 18.0220],
         [ 14.1748],
         [-12.2327],
         [  1.7682],
         [ -1.8642],
         [-12.9622],
         [  0.5952],
         [  0.3353],
         [ -1.3261],
         [ 12.3926],
         [ -7.9529],
         [  3.4371],
         [-16.8843],
         [  

## 定义模型
有100个权重，相应有100个特征

In [7]:
from torch import nn

net = nn.Sequential(nn.Linear(100, 1))

## 初始化模型参数

In [8]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.normal_(0, 0.01)

tensor([0.0106])

## 定义损失模型

In [9]:
loss = nn.MSELoss()

## 定义优化算法
由于一版本损失值下降慢，这里考虑提高学习率为0.04

In [10]:
trainer = torch.optim.SGD(net.parameters(), lr=0.04)

## 训练并监控过程
#这里也考虑适度提高监测范围，以便为后续改进做参考

这里也加大到16

In [11]:
num_epochs = 16
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 1.193121
epoch 2, loss 1.208382
epoch 3, loss 1.207541
epoch 4, loss 1.195382
epoch 5, loss 1.194016
epoch 6, loss 1.180001
epoch 7, loss 1.198387
epoch 8, loss 1.184155
epoch 9, loss 1.193265
epoch 10, loss 1.187266
epoch 11, loss 1.170595
epoch 12, loss 1.200538
epoch 13, loss 1.217809
epoch 14, loss 1.210870
epoch 15, loss 1.193321
epoch 16, loss 1.223500


这里遇到报错
```
Cell In[13], line 4
      2 print('w的估计误差：', true_w - w.reshape(true_w.shape))
      3 b = net[0].bias.data
----> 4 print('b的估计误差：', true_b - b.reshape(true_b.shape))

RuntimeError: shape '[100]' is invalid for input of size 1
```
原因：
`b` 是神经网络的偏置参数，通常是标量（shape `[]` 或 `[1]`，size=1）
`true_b` 被定义为形状 `[100]` 的数组 / 张量，两者维度无法直接相减

解决方案：
将b扩展为true_b的形状

In [12]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b.expand_as(true_b))

w的估计误差： tensor([-0.0496,  0.0284, -0.0095,  0.0506, -0.0487, -0.0179,  0.0337,  0.0099,
         0.0462,  0.0103, -0.0698,  0.0541, -0.0272, -0.0083,  0.0361,  0.0488,
         0.0132, -0.0068,  0.0021,  0.0418,  0.0377, -0.0393, -0.0042, -0.0257,
        -0.0026,  0.0184,  0.0796, -0.0520, -0.0170,  0.0299,  0.0277,  0.0171,
         0.0810,  0.0156,  0.0372, -0.0130, -0.0063, -0.0531, -0.0091,  0.0319,
         0.0259,  0.0438,  0.0659,  0.0292,  0.0055,  0.0076,  0.0180,  0.0262,
        -0.0374, -0.0508,  0.0179,  0.0234,  0.0096,  0.0139, -0.0311,  0.0040,
         0.0442, -0.0108,  0.0066,  0.0651,  0.0322, -0.0161, -0.0060,  0.0126,
         0.0741, -0.0030,  0.0239, -0.0297,  0.0279, -0.0130,  0.0351, -0.0144,
         0.0085,  0.0248, -0.0420, -0.0453,  0.0281,  0.0152,  0.0023,  0.0337,
        -0.0019,  0.0753,  0.0743, -0.0001,  0.0229, -0.0127, -0.0270, -0.0281,
         0.0467, -0.0170, -0.0322,  0.0247, -0.0068,  0.0182,  0.0242, -0.0079,
         0.0202, -0.0549,  0.022

第一版模型误差值
w的估计误差： tensor([-0.0365,  0.0147,  0.0195, -0.0482,  0.0737,  0.0061, -0.0253,  0.0460,
        -0.0137, -0.0162, -0.0311,  0.0146,  0.0147,  0.0155,  0.0005, -0.0067,
         0.0417, -0.0532,  0.0150, -0.0060,  0.0320,  0.0713, -0.0056, -0.1093,
        -0.0404,  0.0063,  0.0023, -0.0008,  0.0392,  0.0221,  0.0069, -0.0320,
        -0.0090,  0.0477, -0.0029,  0.0304,  0.0312, -0.0966, -0.0234, -0.0444,
         0.0048,  0.0411,  0.0519,  0.0331, -0.0080, -0.0720, -0.0994, -0.0168,
         0.0448, -0.0533,  0.0148,  0.0043,  0.0262,  0.0375,  0.0486,  0.0027,
         0.0162,  0.0092,  0.0117,  0.0030, -0.0415, -0.0231,  0.0421, -0.0230,
        -0.0221,  0.0510,  0.0218, -0.0276,  0.0322,  0.0359, -0.0328,  0.0505,
         0.0350, -0.0086,  0.0070,  0.0073,  0.0034,  0.0249, -0.0294,  0.0192,
         0.0514,  0.0709, -0.0017, -0.0943, -0.0350,  0.0418,  0.0118, -0.0014,
        -0.0271, -0.0293,  0.0028,  0.0211, -0.0158, -0.0335,  0.0371, -0.0397,
        -0.0296,  0.0479, -0.0095,  0.0326])
b的估计误差： tensor([-0.0561, -0.5050, -0.6661,  0.8930,  0.4008, -0.9505,  0.0765,  2.6780,
         2.0546,  0.6575, -1.0989, -0.9305, -0.8770, -0.5827, -1.0105,  1.3163,
        -0.5277, -1.2330, -0.2130, -0.2460,  1.8083, -0.3696,  1.8805, -1.5558,
        -0.1420, -0.4572,  1.6652,  0.2477,  0.6272,  0.0858, -2.4407, -0.4614,
         1.3147,  2.4824, -0.1110, -0.9497,  0.9925, -2.3830, -1.1329, -0.1971,
        -0.1680, -0.1179,  0.0830, -0.4045,  1.6900, -2.0702,  1.0407, -0.6841,
        -0.1101,  0.6642, -0.2374,  2.2902,  0.7200, -1.3099, -0.7903,  1.0694,
         0.2521, -1.6064, -1.4621, -0.3737,  0.2395,  0.6590,  1.7287,  0.6078,
         0.3454, -0.2533, -0.6378,  0.8137,  0.9487,  0.0759, -0.2767, -2.0323,
         1.4126,  1.1395,  0.1721,  0.9714, -0.2772, -1.0671, -0.5077,  1.2575,
        -0.0909, -0.2047, -0.0111,  0.2395, -1.0150,  0.6759,  0.1038, -1.0899,
        -0.9807,  0.0453, -0.3143, -1.0716, -0.1950,  0.3526,  0.3375, -0.9365,
         2.4283, -1.0303, -0.7100, -0.0236])
第一版模型总结：10000样本对应100w和b,结果生成误差值相对较大，损失值也降不下来。
可能原因：
1. 第一考虑因素：训练数据不够大
2. 第二考虑因素：模型未调配好
3. 第三考虑因素：未知原因即自己没考虑到的因素
